# W6C1: Taking a transformer apart with your hands

Run every cell from the top. **Everything already works.**

We are not implementing anything from the slides. We are loading a trained transformer and taking it apart. Everything here runs on a laptop in a second or two.

Today you will:

1. Turn a sentence into **one vector per token**, all positions at once.
2. Pull the **attention weights** out and look at what attends to what.
3. Watch the **same word get a different vector** in a different sentence.
4. Then find out what these models do and do not know.

Nothing to submit. Answers are in the last cell.

In [ ]:
# Setup. Run this cell first. The models are cached after the first time.
import warnings
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import transformers
from transformers import AutoTokenizer, AutoModel

warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()
sns.set_theme(style="white")

# DistilBERT: six transformer blocks, twelve heads each. `eager` attention is
# the slower implementation, and the only one that will hand back its weights.
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
encoder = AutoModel.from_pretrained("distilbert-base-uncased",
                                    attn_implementation="eager")
encoder.eval()

print("parameters:", f"{sum(p.numel() for p in encoder.parameters()):,}")
print("blocks    :", encoder.config.num_hidden_layers)
print("heads     :", encoder.config.num_attention_heads)
print("width     :", encoder.config.dim)

## Part 1. A sentence goes in, one vector per token comes out


First the sentence is split into tokens, and the tokenizer has its own ideas
about where the boundaries are.

Then every token becomes a vector. Not one vector for the sentence: **one per
token**, and every one of them has seen every other one.


In [ ]:
sentence = "the bank raised interest rates again"

batch = tokenizer(sentence, return_tensors="pt")
tokens = tokenizer.convert_ids_to_tokens(batch["input_ids"][0])
print("tokens:", tokens)
print()

with torch.no_grad():
    output = encoder(**batch, output_attentions=True)

print("last_hidden_state:", tuple(output.last_hidden_state.shape),
      " (batch, tokens, width)")
print()
print("Every token now has 768 numbers, and they were all computed at the same")
print("time. There is no loop over positions anywhere in this model.")

In [ ]:
# ================== TRY IT 1 ==================
# Tokenize a sentence containing an unusual word, such as
# "the paleontologist tokenized it". How many tokens does the odd word
# turn into, and what are they?
# ==============================================


## Part 2. Look at the attention weights


The slides drew attention as a matrix of weights: for each token, how much it
takes from every other token. That matrix is a real array you can read.

DistilBERT has six blocks of twelve heads, so there are 72 of these matrices for
any sentence. Here is one.


In [ ]:
# output.attentions is one entry per block: (batch, heads, tokens, tokens)
print("blocks:", len(output.attentions), " each", tuple(output.attentions[0].shape))
print()

LAYER, HEAD = 4, 7
weights = output.attentions[LAYER][0, HEAD].numpy()

plt.figure(figsize=(5.5, 4.5))
sns.heatmap(weights, xticklabels=tokens, yticklabels=tokens,
            cmap="Reds", vmin=0, vmax=1, square=True, cbar_kws={"shrink": .7})
plt.title(f"block {LAYER}, head {HEAD}")
plt.ylabel("this token...")
plt.xlabel("...attends to this one")
plt.tight_layout()
plt.show()

print("Each ROW sums to 1: it is a softmax over the tokens.")
print("row sums:", weights.sum(axis=1).round(3))

In [ ]:
# ================== TRY IT 2 ==================
# Look at two or three other heads by changing LAYER and HEAD.
# Do they all do the same thing?
# ==============================================


## Part 3. The same word, two different vectors


A fixed embedding table gives a word one vector for ever. That is why `apple`
comes out as fruit and company blended together, and why nothing can tell the two
apart.

A transformer builds the vector **from the sentence**, so the same word in two
sentences comes out differently.


In [ ]:
def vector_for(word, sentence):
    """The output vector for one word, in the context of one sentence."""
    batch = tokenizer(sentence, return_tensors="pt")
    tokens = tokenizer.convert_ids_to_tokens(batch["input_ids"][0])
    with torch.no_grad():
        hidden = encoder(**batch).last_hidden_state[0]
    return hidden[tokens.index(word)]


def cosine(first, second):
    return float(torch.nn.functional.cosine_similarity(first, second, dim=0))


money = vector_for("bank", "he deposited the cheque at the bank")
river = vector_for("bank", "he sat on the grassy bank of the river")
other = vector_for("bank", "the bank approved her mortgage")

print(f"  money bank  vs  river bank : {cosine(money, river):.3f}")
print(f"  money bank  vs  other bank : {cosine(money, other):.3f}")
print()
print("Same word, same spelling, same embedding row on the way in.")
print("Different vectors on the way out, because the sentence is different.")

In [ ]:
# ================== TRY IT 3 ==================
# Try it on another word with two meanings, such as "light" or "bat".
# Does the gap look the same?
# ==============================================



---

## Your turn: find out what it knows

Two models, two helper functions, and almost no code to write. **The encoder**
fills in a blank. **The decoder** continues a prompt. Everything else is choosing
sentences and reading numbers.

**Teams of three, about twenty-five minutes.** Write down what you find, because
the last ten minutes are each team reporting the most surprising thing.

<img src="images/activity-probe.png" width="640">

**One rule:** a claim about the model comes with the prompt that produced it and
the numbers it returned. "It is biased" is not a finding. "`the doctor finished
[MASK] shift` gives *his* 0.44 and *her* 0.10" is a finding.


In [ ]:
# GIVEN. Two functions. You will not need to edit this cell.
from transformers import AutoModelForMaskedLM, AutoModelForCausalLM

filler = AutoModelForMaskedLM.from_pretrained("distilbert-base-uncased")
filler.eval()

gpt_tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
gpt = AutoModelForCausalLM.from_pretrained("distilgpt2")
gpt.eval()


def fill_mask(text, how_many=5):
    """Ask the ENCODER what belongs in the [MASK]. Returns word, probability."""
    batch = tokenizer(text, return_tensors="pt")
    spot = int((batch["input_ids"][0] == tokenizer.mask_token_id).nonzero()[0])
    with torch.no_grad():
        scores = filler(**batch).logits[0, spot]
    best = torch.softmax(scores, -1).topk(how_many)
    return [(tokenizer.decode([i]).strip(), round(float(v), 3))
            for v, i in zip(best.values, best.indices)]


def complete(prompt, temperature=0.8, words=25, seed=0):
    """Ask the DECODER to continue the prompt."""
    torch.manual_seed(seed)
    ids = gpt_tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        out = gpt.generate(**ids, max_new_tokens=words, do_sample=True,
                           temperature=temperature, top_k=50,
                           pad_token_id=gpt_tokenizer.eos_token_id)
    return gpt_tokenizer.decode(out[0], skip_special_tokens=True)


print(fill_mask("the [MASK] sat on the mat ."))
print()
print(complete("The best thing about university is"))

In [ ]:
# ================== YOUR TURN 1 ==================
# Probe the encoder. Pick at least three of the six questions on the
# slide and write the prompts that answer them.
#
# Keep the pairs that differ by one word: that is what makes a result
# mean something.
#
# Expected: a handful of results you can defend. Some of them will surprise
#           you: it handles agreement across a distracting noun, and it does not
#           handle "not" at all.
# =================================================
PROMPTS = [
    "the keys to the cabinet [MASK] on the table .",
    "the key to the cabinets [MASK] on the table .",
    # <-- your prompts here
]

for prompt in PROMPTS:
    print(prompt)
    print("   ", fill_mask(prompt))
    print()

In [ ]:
# ================== YOUR TURN 2 ==================
# Probe the decoder. Run the same prompt at three temperatures, and
# find a prompt where the continuation says something you would not want
# a product to say.
#
# Expected: temperature 0.2 repeats itself and gets stuck; 1.5 stops making
#           sense. And a bland prompt about a person's job will happily assume
#           which person.
# =================================================
PROMPT = "The nurse walked into the room and"

for temperature in [0.2, 0.8, 1.5]:
    print(f"--- temperature {temperature}")
    print(complete(PROMPT, temperature=temperature))
    print()

## Answers

Try each task before reading.

In [ ]:
# TRY IT 1
#   tokenizer.convert_ids_to_tokens(tokenizer("the paleontologist tokenized it")["input_ids"])
#   "paleontologist" becomes pale ##onto ##logist and "tokenized" becomes
#   token ##ized. The model has a fixed vocabulary of about 30,000 pieces, so a
#   rare word is spelled out of common fragments. Nothing is ever unknown, which
#   is why there is no <unk> anywhere in this notebook.

# TRY IT 2
#   No. Heads specialise, and most of them are not interpretable. Some attend
#   almost entirely to [SEP] or to the previous token, which is a real and much
#   studied finding, not a bug. Look at enough of them and you will find one
#   that tracks something recognisable, and it is a mistake to conclude the
#   model "understands" that thing because of it.

# TRY IT 3
#   A smaller gap than you expect. Measured for "bank": the money and river
#   senses sit at 0.563, and two different money sentences sit at 0.785. So the
#   vector does move with the sentence, which is the whole point, but the two
#   senses are not cleanly separated and the same sense is not identical either.

# YOUR TURN 1  (measured, distilbert-base-uncased)
#   the keys to the cabinet [MASK] on the table
#       are 0.137, rested 0.099, were 0.060, sat 0.052
#   the key to the cabinets [MASK] on the table
#       rested 0.136, sat 0.054, was 0.041, lay 0.037
#     Agreement is correct in both, ACROSS an intervening noun of the other
#     number. Nobody taught it that rule.
#
#   the capital of australia is [MASK]      canberra 0.552   correct
#   the capital of france is [MASK]         marseille 0.143, nantes 0.090,
#                                           toulouse 0.088, paris 0.086
#     It is not a database. It has the shape of the answer, French cities, and
#     the wrong one on top.
#
#   a robin is a [MASK]                     bird 0.160
#   a robin is not a [MASK]                 robin 0.074, bird 0.051
#     "not" barely changes the answer. Masked language models are known to be
#     poor at negation, and this is the cheapest demonstration of it.
#
#   the doctor finished [MASK] shift        his 0.439, the 0.224, her 0.101
#   the nurse finished [MASK] shift         her 0.610, the 0.140, his 0.041
#     Same sentence, one word different, and the pronoun flips. This is the
#     embedding bias from earlier in the course, in a model 300 times bigger.

# YOUR TURN 2
#   Low temperature repeats and loops; high temperature loses the thread. The
#   nurse/doctor prompt reliably assigns a gender the prompt never gave it.
#   Worth saying: nothing here was fine-tuned or filtered, which is exactly why
#   the raw behaviour is visible. Everything after Week 9 is about what you put
#   on top of a model like this before anyone is allowed near it.

# The three things worth carrying out of today:
#   1. A transformer turns n tokens into n vectors at once. No recurrence, no
#      loop over positions, and that is why it trains on long sequences.
#   2. The vector for a word is built from its sentence, so "bank" has no single
#      meaning stored anywhere.
#   3. What the model knows and what it merely patterns are different things,
#      and one afternoon of prompting is enough to tell them apart.